# Chapter 01: Auto-Merge & The Airlock Principle (Reference)

## Learning Objectives

- State the Airlock Principle in one sentence
- Explain why gates must fail closed, with a real prior-art counterexample
- Construct a Decision from three GateResults
- Run three airlock scenarios and read their decision tables

## Setup

The next cell sets up reproducibility, the `PRA_MODE` fixture/live toggle, and inserts the repo root onto `sys.path` so `pr_automerge` is importable. You should see `PRA_MODE = 'fixture'` printed (unless you've set it to `live`).

In [2]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

# Reproducibility -- always set before any stochastic operation
RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

# PRA_ environment variables -- fixture mode by default so this notebook runs
# identically for every reader. Set PRA_MODE=live and PRA_REPO=owner/name before
# starting Jupyter to run this against the real sandbox repo instead.
PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")
OUTPUT_DIR = Path(os.environ.get("PRA_OUTPUT_DIR", "output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run in live mode"

print(f"PRA_MODE = {PRA_MODE!r}")
print(f"RANDOM_STATE = {RANDOM_STATE}")

PRA_MODE = 'fixture'
RANDOM_STATE = 42


## 1. Construct Three Stub Gates

The next cell builds three `GateResult`s by hand -- one per gate -- so you can see the shared model before any real GitHub calls exist. You should see three PASS rows.

In [3]:
from pr_automerge.models import GateResult, GateStatus

gates = [
    GateResult("gate1_repo_readiness", GateStatus.PASS, "repo is ready"),
    GateResult("gate2_pr_health", GateStatus.PASS, "CI is green"),
    GateResult("gate3_risk_scoring", GateStatus.PASS, "diff is small"),
]
for g in gates:
    print(g.gate, g.status.value, g.passed)

gate1_repo_readiness pass True
gate2_pr_health pass True
gate3_risk_scoring pass True


## 2. Compose a Decision

The next cell composes those three gate results into one `Decision`. This matters because the merge verdict is never computed ad hoc -- it's always `all(g.passed for g in gates)`. You should see `merge=True`.

In [4]:
from pr_automerge.models import Decision

decision = Decision(pr_number=1, merge=all(g.passed for g in gates), gates=gates)
print(f"merge={decision.merge}")
print(f"failed_gates={decision.failed_gates()}")

merge=True
failed_gates=[]


## 3. Run the Full Three-Scenario Demo

The next cell runs `run_stub_airlock` from `labs/lab_01_airlock_principle.py` across all three scenarios from the lab script's `main()`. You should see one MERGE and two HOLD verdicts, matching Chapter 01 Section 4's table.

In [5]:
from labs.lab_01_airlock_principle import run_stub_airlock

for ready, healthy, safe in [
    (True, True, True),
    (True, False, True),
    (False, True, True),
]:
    d = run_stub_airlock(1, ready=ready, healthy=healthy, safe_size=safe)
    print(f"ready={ready} healthy={healthy} safe={safe} -> merge={d.merge}")

ready=True healthy=True safe=True -> merge=True
ready=True healthy=False safe=True -> merge=False
ready=False healthy=True safe=True -> merge=False


## Takeaways & Next Steps

This notebook's takeaways are the numbers you just produced above, not abstract claims -- re-read the printed output from each section before moving on.

In [6]:
print(
    "Re-run this notebook with PRA_MODE=live to see it against the real sandbox repo."
)

Re-run this notebook with PRA_MODE=live to see it against the real sandbox repo.


---

📖 **Reading companion:** [Chapter 01: Auto-Merge & The Airlock Principle](../learning_modules/chapter_01_airlock_principle.md)
🔬 **Try it live:** this chapter's lab is fully offline (no `gh` calls), so `PRA_MODE=live` changes nothing here — nothing to re-run against a real repo.
